In [2]:
# pip 최신버전 업그레이드
!python -m pip install --upgrade pip

In [ ]:
# neo4j 관련 라이브러리 설치
%pip install neo4j langchain-neo4j


   -------- ------------------------------- 1/5 [neo4j]
   -------- ------------------------------- 1/5 [neo4j]
   -------- ------------------------------- 1/5 [neo4j]
   -------- ------------------------------- 1/5 [neo4j]
   -------- ------------------------------- 1/5 [neo4j]
   -------- ------------------------------- 1/5 [neo4j]
   -------- ------------------------------- 1/5 [neo4j]
   -------- ------------------------------- 1/5 [neo4j]
   -------- ------------------------------- 1/5 [neo4j]
   -------- ------------------------------- 1/5 [neo4j]
   -------- ------------------------------- 1/5 [neo4j]
   -------- ------------------------------- 1/5 [neo4j]
   -------- ------------------------------- 1/5 [neo4j]
   ---------------- ----------------------- 2/5 [json-repair]
   ---------------- ----------------------- 2/5 [json-repair]
   ---------------- ----------------------- 2/5 [json-repair]
   ------------------------ --------------- 3/5 [neo4j-graphrag]
   -----------------

In [4]:
import os
import pandas as pd 
from dotenv import load_dotenv
from neo4j import GraphDatabase

In [5]:
load_dotenv()  # .env 파일을 환경변수로 등록

NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE', "neo4j")  # 기본값 neo4j

In [ ]:
# neo4j Driver 생성
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

driver.verify_connectivity()  # 실제로 연결됐는지 확인

print('Neo4j Driver 연결 성공')

Neo4j Driver 연결 성공


In [8]:
# Python에서 Cypher 실행
query = """
MATCH (student:Student)
RETURN
    student.student_id AS student_id,
    student.name AS name,
    student.age AS age
ORDER BY student.student_id
"""

# 쿼리 결과 반환 (실제 조회 결과 행들, 쿼리 실행 요약 정보, 결과 컬럼명)
records, summary, keys = driver.execute_query(query, database=NEO4J_DATABASE)

student = [record.data() for record in records]

student

[{'student_id': 1, 'name': '홍길동', 'age': 22},
 {'student_id': 2, 'name': '김영희', 'age': 24},
 {'student_id': 3, 'name': '이민수', 'age': 25},
 {'student_id': 4, 'name': '박서연', 'age': 28},
 {'student_id': 5, 'name': '최준호', 'age': 23}]

In [9]:
student_df = pd.DataFrame(student)

student_df

,student_id,name,age
0,1,홍길동,22
1,2,김영희,24
2,3,이민수,25
3,4,박서연,28
4,5,최준호,23


In [12]:
#######~깃허브~

In [ ]:
# 파라미터를 이용한 조회
query = """
MATCH (student:Student {name: $student_name})-[enroll:ENROLLEN_IN]->(course:Course)
RETURN
    student.name AS student_name,
    enroll.name AS score,
    course.course_id AS course_id
ORDER BY course_id;
"""

# 쿼리 결과 반환 (실제 조회 결과 행들, 쿼리 실행 요약 정보, 결과 컬럼명)
records, summary, keys = driver.execute_query(
    query, 
    student_name = '홍길동',
    database=NEO4J_DATABASE
)

result = [record.data() for record in records]

result

result_df = pd.DataFrame(result)

result_df

Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `ENROLLEN_IN` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=2, column=55, offset=55>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 55, 'line': 2, 'column': 55}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (student:Student {name: $student_name})-[enroll:ENROLLEN_IN]->(course:Course)\nRETURN\n    student.name AS student_name,\n    enroll.name AS score,\n    course.course_id AS course_id\nORDER BY course_id\n'


""


In [ ]:
# 여러 단계의 관계 조회
query = """
MATCH (student:Student)-[:ENROLLED_IN]->(course:Course)<-[:TEACHES]-(instructor:Instructor)
MATCH (course)-[:BELONGS_TO]->(category:Category)
RETURN
    student.name AS student_name,
    course.name AS course_name,
    instructor.name AS instructor_name,
    category.name AS category_name,
    student.student_id AS student_id,
    course.course_id AS course_id
ORDER BY student_id, course_id;
"""

# 쿼리 결과 반환 (실제 조회 결과 행들, 쿼리 실행 요약 정보, 결과 컬럼명)
records, summary, keys = driver.execute_query(query, database=NEO4J_DATABASE)

result = [record.data() for record in records]

result_df = pd.DataFrame(result)

result_df

,student_name,course_name,instructor_name,category_name,student_id,course_id
0,홍길동,Python,Capybara,프로그래밍,1,101
1,김영희,백이번,Capybara,데이터베이스,2,102
2,이민수,Python,Capybara,프로그래밍,3,101
3,최준호,백이번,Capybara,데이터베이스,5,102


In [ ]:
# 여러 단계의 관계 조회
query = """
MATCH (student:Student)-[:ENROLLED_IN]->(course:Course)
RETURN
    course.name AS course_name,
    count(student) AS student_count
ORDER BY student_count DESC, course_name;
"""

# 쿼리 결과 반환 (실제 조회 결과 행들, 쿼리 실행 요약 정보, 결과 컬럼명)
records, summary, keys = driver.execute_query(query, database=NEO4J_DATABASE)

result = [record.data() for record in records]

result_df = pd.DataFrame(result)

result_df

,course_name,student_count
0,Data a,2
1,Python,2
2,백삼,2
3,백이번,2
4,백오,1
5,백육,1


In [16]:
driver.close()